# Transfer Learning with CNNs — Guided Fashion-MNIST + CIFAR-10 Challenge

**References**
- Course CNN notebook: https://colab.research.google.com/github/serivan/DeepLearning/blob/master/PyTorch/12_deep_computer_vision_with_cnns.ipynb
- Previous live CNN workflow: Fashion-MNIST convolutional classifier in PyTorch
- TorchVision pretrained models: https://docs.pytorch.org/vision/main/models.html
- Optuna: https://optuna.readthedocs.io/

## Learning goals
By the end of this lab you should be able to:
1. explain the difference between **training from scratch**, **feature extraction**, and **fine-tuning**;
2. adapt a pretrained CNN to a new classification task;
3. decide which parameters should be trainable and assign different learning rates to different parts of a network;
4. design and train your own CNN on CIFAR-10;
5. compare your CNN with transfer learning using a pretrained model of your choice;
6. use **Optuna** to tune the most relevant hyperparameters systematically;
7. retrain, evaluate, save, and reload the final selected model.

> The notebook is divided into a short **guided live section on Fashion-MNIST** and an **independent CIFAR-10 challenge**. The test set must not be used for hyperparameter selection.


In [ ]:
import copy
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
pin_memory = device.type == "cuda"

print("PyTorch:", torch.__version__)
print("device:", device)


# Part A — Guided live: transfer learning on Fashion-MNIST

The previous CNN lab trained all convolutional filters **from scratch**. Here we keep Fashion-MNIST so that the dataset is familiar, but we change the learning strategy:

\[
\text{ImageNet pretrained CNN} \rightarrow \text{replace head} \rightarrow \text{freeze} \rightarrow \text{train head} \rightarrow \text{fine-tune selected layers}
\]

The purpose is not to obtain the best possible Fashion-MNIST score. Fashion-MNIST is intentionally very different from ImageNet; this makes **domain mismatch** visible and gives us something meaningful to discuss.


## A1 — Recall the CNN trained from scratch

A compact CNN previously studied on Fashion-MNIST can be written as one `nn.Sequential` pipeline. We will not retrain it here; use it only as a reference for the conceptual comparison.

**Discuss before running:**
- where are the features learned in this model?
- which parameters start from random initialization?
- if we move to a pretrained network, what changes?


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 64),
            nn.ReLU(),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        return self.net(x)

reference_cnn = SmallCNN()
print(reference_cnn)
print("parameters:", f"{count_parameters(reference_cnn)[0]:,}" if 'count_parameters' in globals() else "run the helper cell below")


## A2 — Training and evaluation helpers

We deliberately use **two separate functions**:
- `train_one_epoch()` updates parameters;
- `evaluate()` never computes gradients or updates parameters.

This keeps the distinction explicit during the live session.


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    """Train for one epoch and return mean loss and accuracy."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for X, y in loader:
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        total += y.numel()

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    """Evaluate without updating model parameters."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for X, y in loader:
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(X)
        loss = criterion(logits, y)

        running_loss += loss.item() * X.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        total += y.numel()

    return running_loss / total, correct / total


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


## A3 — Pretrained ResNet-18: input requirements

We use **ResNet-18 pretrained on ImageNet**. The pretrained weights expect an ImageNet-style RGB input and their associated preprocessing.

Fashion-MNIST images are grayscale, so we must first convert them to RGB.

**Predict before coding:**
1. why can we not simply use the original 28×28 grayscale tensor?
2. what happens to computational cost when a 28×28 image is resized to an ImageNet-style input?
3. should the original ImageNet 1000-class head be kept?


In [ ]:
# TODO A1 — select the default pretrained ResNet-18 weights.
# weights = ...

# TODO A2 — build the transform:
# 1) convert Fashion-MNIST grayscale PIL images to RGB;
# 2) apply the preprocessing associated with the pretrained weights.
# transfer_transform = transforms.Compose([
#     ...,
#     ...,
# ])

# TODO A3 — create Fashion-MNIST train and test datasets using transfer_transform.
# fashion_train_full = ...
# fashion_test = ...

# Fixed train/validation split.
g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(60_000, generator=g)
valid_idx = perm[:5_000].tolist()
train_idx = perm[5_000:].tolist()

# Keep the live experiment short.
LIVE_FAST = True
if LIVE_FAST:
    train_idx_live = train_idx[:4_000]
    valid_idx_live = valid_idx[:1_000]
    test_idx_live = list(range(1_000))
else:
    train_idx_live = train_idx
    valid_idx_live = valid_idx
    test_idx_live = list(range(10_000))

# TODO A4 — create Subset objects and DataLoaders.
# fashion_train = ...
# fashion_valid = ...
# fashion_test_live = ...
# fashion_train_loader = ...
# fashion_valid_loader = ...
# fashion_test_loader = ...

# TODO A5 — inspect one batch. Expected input shape for ResNet: [B, 3, H, W].


## A4 — Turn ResNet-18 into a Fashion-MNIST feature extractor

Feature-extraction workflow:
1. load pretrained weights;
2. freeze pretrained parameters;
3. replace the original classification head;
4. train only the new head.

**Important:** freezing parameters saves gradient/update work, but the forward pass through the backbone is still required.


In [ ]:
# TODO A6 — load pretrained ResNet-18.
# transfer_model = ...

# TODO A7 — freeze every pretrained parameter.
# for p in transfer_model.parameters():
#     ...

# TODO A8 — replace the 1000-class ImageNet head with a 10-class head.
# in_features = ...
# transfer_model.fc = ...
# transfer_model = transfer_model.to(device)

# TODO A9 — inspect total and trainable parameter counts.
# total_params, trainable_params = ...
# print(...)

# TODO A10 — run a forward pass on a few images and check logits.shape.


## A5 — Train only the new head

Use `CrossEntropyLoss`. The optimizer must initially receive **only the parameters of the new classification head**.

After the short run, compare training and validation accuracy. Do not expect ImageNet → Fashion-MNIST transfer to be perfect: the source and target domains are very different.


In [ ]:
# TODO A11 — define criterion and an optimizer for transfer_model.fc only.
# criterion_tl = ...
# optimizer_head = ...

# TODO A12 — train the head for 1–2 epochs using train_one_epoch() and evaluate().
# LIVE_EPOCHS = ...
# for epoch in range(LIVE_EPOCHS):
#     ...

# TODO A13 — evaluate the current model on fashion_test_loader.
# Keep this as a demonstration; in the CIFAR-10 challenge the test set will be reserved for the final model.


## A6 — From feature extraction to partial fine-tuning

Now allow the last residual stage (`layer4`) to adapt.

Use **discriminative learning rates**:
- a small learning rate for pretrained `layer4`;
- a larger learning rate for the newly initialized `fc` head.

**Discuss:** why is a smaller learning rate usually safer for pretrained parameters?


In [ ]:
# TODO A14 — unfreeze layer4.
# for p in transfer_model.layer4.parameters():
#     ...

# TODO A15 — build an optimizer with two parameter groups.
# optimizer_ft = torch.optim.AdamW([
#     {"params": ..., "lr": 1e-5},
#     {"params": ..., "lr": 1e-3},
# ], weight_decay=1e-4)

# TODO A16 — print the new trainable parameter count.

# OPTIONAL LIVE STEP — run one short fine-tuning epoch and compare validation accuracy.


### Live checkpoint
Be ready to explain these four statements in your own words:
1. **Training from scratch:** all useful visual filters must be learned from the target dataset.
2. **Feature extraction:** the pretrained representation is fixed and only the new head is trained.
3. **Fine-tuning:** some pretrained layers are allowed to adapt to the target domain.
4. **Domain similarity matters:** transfer is generally easier when source and target visual statistics/tasks are related.


# Part B — Independent challenge: CIFAR-10

Work individually or in a small group according to the course instructions.

You must complete three experiments:

1. **Your own CNN** — design and train a CNN from scratch on CIFAR-10.
2. **Transfer learning** — choose a pretrained TorchVision model and adapt it to CIFAR-10.
3. **Optuna tuning** — tune meaningful architecture/training hyperparameters of your own CNN, analyze the search, retrain the best configuration, evaluate it once on the test set, and save it.

## Required evidence in the notebook
For each experiment, keep the code, outputs, and a short interpretation. At minimum report:
- architecture / selected backbone;
- total and trainable parameter counts;
- validation accuracy;
- training time or qualitative computational-cost observation;
- design choices and what you learned from the result.

For Optuna also report:
- search space;
- best trial and best parameters;
- completed / pruned / early-stopped trials;
- parameter importance or another systematic analysis of hyperparameter behavior;
- final test accuracy;
- saved checkpoint path.


## B1 — CIFAR-10 data

CIFAR-10 contains RGB 32×32 images from 10 classes. We create separate training and validation dataset objects so that data augmentation is applied only to training samples.

The code below is intentionally mostly prepared. Inspect it and explain the role of `RandomCrop`, `RandomHorizontalFlip`, and normalization.


In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

cifar_train_aug = datasets.CIFAR10(
    root="data", train=True, download=True, transform=train_transform
)
cifar_train_eval = datasets.CIFAR10(
    root="data", train=True, download=True, transform=eval_transform
)
cifar_test = datasets.CIFAR10(
    root="data", train=False, download=True, transform=eval_transform
)

cifar_classes = cifar_train_aug.classes

g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(len(cifar_train_aug), generator=g)
valid_idx = perm[:5_000].tolist()
train_idx = perm[5_000:].tolist()

cifar_train = Subset(cifar_train_aug, train_idx)
cifar_valid = Subset(cifar_train_eval, valid_idx)

BATCH_SIZE = 128
cifar_train_loader = DataLoader(
    cifar_train, batch_size=BATCH_SIZE, shuffle=True,
    pin_memory=pin_memory, num_workers=2
)
cifar_valid_loader = DataLoader(
    cifar_valid, batch_size=BATCH_SIZE, shuffle=False,
    pin_memory=pin_memory, num_workers=2
)
cifar_test_loader = DataLoader(
    cifar_test, batch_size=BATCH_SIZE, shuffle=False,
    pin_memory=pin_memory, num_workers=2
)

X, y = next(iter(cifar_train_loader))
print("batch:", X.shape, y.shape)
print("classes:", cifar_classes)


## B2 — Experiment 1: design your own CNN

Design a model rather than copying the Fashion-MNIST architecture unchanged.

### Minimum requirements
Your network must:
- accept `[B, 3, 32, 32]` inputs;
- contain at least two convolutional stages;
- reduce spatial resolution using pooling or stride;
- produce 10 logits;
- expose at least **three configurable choices** that will later be tunable by Optuna (for example base channels, dropout, BatchNorm, activation).

You may use `AdaptiveAvgPool2d` to avoid hard-coding a flattened spatial size.

### Before training
Write 3–5 lines explaining your architecture and why you chose it.


In [ ]:
class StudentCNN(nn.Module):
    def __init__(
        self,
        n_classes=10,
        base_channels=32,
        dropout=0.2,
        use_batchnorm=True,
        activation="relu",
    ):
        super().__init__()

        # TODO B1 — construct YOUR CNN.
        # Keep a single self.net = nn.Sequential(...) if possible.
        # The arguments above should actually influence the architecture.
        self.net = nn.Sequential(
            # ...
        )

    def forward(self, x):
        # TODO B2 — return logits.
        pass


# TODO B3 — instantiate your baseline configuration and verify output shape [B, 10].
# student_model = ...
# with torch.no_grad():
#     print(student_model(X[:4].to(device)).shape)
# print("parameters:", ...)


In [ ]:
# TODO B4 — train your baseline CNN from scratch.
# Suggested starting point: AdamW, lr=1e-3, weight_decay=1e-4, 3–5 epochs.
# Record validation accuracy and elapsed time.

# baseline_criterion = ...
# baseline_optimizer = ...
# BASELINE_EPOCHS = ...
# t0 = time.perf_counter()
# for epoch in range(BASELINE_EPOCHS):
#     ...
# baseline_seconds = time.perf_counter() - t0
# baseline_val_loss, baseline_val_acc = ...


### B2 reflection
Answer briefly:
1. Which architectural choice had the largest effect on parameter count?
2. Where does the model lose spatial resolution?
3. What evidence suggests underfitting or overfitting?
4. What would you change before increasing the number of epochs?


## B3 — Experiment 2: transfer learning with a model of your choice

Choose **one pretrained CNN** from TorchVision. Recommended choices for this lab:

| Model | TorchVision builder | Classification head |
|---|---|---|
| ResNet-18 | `models.resnet18` | `model.fc` |
| EfficientNet-B0 | `models.efficientnet_b0` | `model.classifier[-1]` |
| MobileNetV3-Small | `models.mobilenet_v3_small` | `model.classifier[-1]` |

You may choose another TorchVision classification model, but you must identify correctly:
- its default pretrained weights;
- its required input preprocessing;
- its classification head;
- a sensible last block for partial fine-tuning.

### Required comparison
1. **Feature extraction:** freeze the pretrained backbone and train only the new head.
2. **Partial fine-tuning:** unfreeze the final block and use a lower learning rate for that block than for the head.

Compare the two approaches using **validation**, not test, accuracy.


In [ ]:
BACKBONE_NAME = "resnet18"  # TODO B5 — choose your model.


def build_transfer_model(name, n_classes=10):
    """Return model, weights, head module, and last block to fine-tune."""
    # TODO B6 — implement the branch corresponding to your chosen backbone.
    # You may keep only one branch if you clearly document your choice.
    raise NotImplementedError


# TODO B7 — build the pretrained model and obtain its weights/head/last block.
# transfer_model, transfer_weights, transfer_head, transfer_last_block = ...

# TODO B8 — freeze the backbone and make sure the replacement head is trainable.

# TODO B9 — recreate CIFAR-10 datasets using transfer_weights.transforms().
# IMPORTANT: pretrained ImageNet weights typically resize the 32x32 CIFAR images.
# Use the SAME train_idx/valid_idx defined above.

# TODO B10 — build transfer-learning DataLoaders and inspect one batch.


In [ ]:
# TODO B11 — feature extraction:
# - CrossEntropyLoss
# - optimizer over transfer_head.parameters() only
# - train for a short run
# - report validation accuracy and trainable parameter count

# TODO B12 — partial fine-tuning:
# - unfreeze transfer_last_block
# - optimizer with two parameter groups
#   * lower lr for the pretrained block
#   * higher lr for the head
# - run a short fine-tuning phase
# - report validation accuracy and trainable parameter count


### B3 reflection
Discuss:
1. Why is ImageNet → CIFAR-10 transfer conceptually more natural than ImageNet → Fashion-MNIST?
2. Did feature extraction or partial fine-tuning work better in your run?
3. How much did the pretrained preprocessing increase input size and computational cost?
4. Is the model with the highest validation accuracy necessarily the best engineering choice? Consider memory, latency, and parameter count.


## B4 — Experiment 3: systematic hyperparameter tuning with Optuna

Tune **your StudentCNN**, not the test set.

To keep the search computationally manageable:
- use a fixed subset of the training and validation data during the Optuna study;
- keep each trial short;
- use both **Optuna pruning** and **per-trial early stopping**;
- retrain the best configuration after the search.

### Minimum search space
Include at least:
- one model-capacity parameter (e.g. `base_channels`);
- one regularization parameter (`dropout`, BatchNorm, or weight decay);
- activation or another architecture choice;
- optimizer;
- learning rate;
- weight decay;
- momentum when SGD is selected.

You may add parameters, but remember that larger search spaces require more trials.


In [ ]:
# Install Optuna in Colab if necessary.
try:
    import optuna
except ImportError:
    %pip -q install optuna
    import optuna

from optuna.importance import get_param_importances


In [ ]:
# Fixed subsets for faster Optuna trials.
SEARCH_TRAIN_N = 12_000
SEARCH_VALID_N = 3_000

search_train = Subset(cifar_train_aug, train_idx[:SEARCH_TRAIN_N])
search_valid = Subset(cifar_train_eval, valid_idx[:SEARCH_VALID_N])

search_train_loader = DataLoader(
    search_train, batch_size=BATCH_SIZE, shuffle=True,
    pin_memory=pin_memory, num_workers=2
)
search_valid_loader = DataLoader(
    search_valid, batch_size=BATCH_SIZE, shuffle=False,
    pin_memory=pin_memory, num_workers=2
)

SEARCH_EPOCHS = 4
EARLY_STOPPING_PATIENCE = 2
EARLY_STOPPING_MIN_DELTA = 1e-3
N_TRIALS = 12


In [ ]:
def build_student_model_from_config(config):
    # TODO B13 — rebuild YOUR StudentCNN from a parameter dictionary.
    # This function is needed both by Optuna and by final retraining.
    raise NotImplementedError


def make_optimizer(model, config):
    # TODO B14 — create AdamW or SGD according to config.
    # Use momentum only when SGD is selected.
    raise NotImplementedError


def objective(trial):
    # TODO B15 — define a meaningful search space.
    config = {
        # "base_channels": trial.suggest_categorical(...),
        # "dropout": trial.suggest_float(...),
        # "use_batchnorm": trial.suggest_categorical(...),
        # "activation": trial.suggest_categorical(...),
        # "optimizer": trial.suggest_categorical(...),
        # "lr": trial.suggest_float(..., log=True),
        # "weight_decay": trial.suggest_float(..., log=True),
    }

    # TODO B16 — conditionally add momentum when optimizer == "sgd".

    # TODO B17 — create model, criterion, optimizer.

    best_val_acc = 0.0
    best_epoch = -1
    epochs_without_improvement = 0
    early_stopped = False

    for epoch in range(SEARCH_EPOCHS):
        # TODO B18 — train one epoch and evaluate on search_valid_loader.
        # val_acc = ...

        # TODO B19 — update best_val_acc and early-stopping counter.

        # Report an intermediate result to Optuna.
        # TODO B20 — trial.report(...)

        # Optuna pruning compares this trial with the study history.
        # TODO B21 — if trial.should_prune(): raise optuna.TrialPruned()

        # Per-trial early stopping only looks at this trial's validation history.
        # TODO B22 — stop if no meaningful improvement for PATIENCE epochs.
        pass

    trial.set_user_attr("best_epoch", best_epoch)
    trial.set_user_attr("early_stopped", early_stopped)
    return best_val_acc


In [ ]:
# TODO B23 — create and run the study.
# Recommended pruner: MedianPruner with a short warm-up.
# study = optuna.create_study(...)
# study.optimize(objective, n_trials=N_TRIALS)

# print("best validation accuracy:", study.best_value)
# print("best parameters:", study.best_params)


## B5 — Analyze the Optuna study

Do not stop at `study.best_params`. Analyze **behavior**, not just the winner.

Required analysis:
1. show the best trials in a DataFrame;
2. count completed and pruned trials;
3. count trials that ended through your own early stopping logic;
4. compute parameter importance;
5. discuss at least **two relationships** between hyperparameters and validation performance.

Examples of questions:
- does a larger model always help?
- is strong dropout useful for every channel width?
- does the best learning-rate region depend on optimizer choice?
- does BatchNorm change the useful learning-rate range?


In [ ]:
# TODO B24 — inspect study.trials_dataframe().
# trials_df = ...
# display(...)

# TODO B25 — completed/pruned/early-stopped counts.

# TODO B26 — compute and display parameter importance.
# importances = get_param_importances(study)
# ...

# TODO B27 — add one plot or grouped table that helps explain hyperparameter behavior.


## B6 — Retrain the best configuration and evaluate the test set once

Now the model-selection process is complete. Rebuild a **fresh** StudentCNN using `study.best_params`, retrain it on the original training split, monitor validation performance, restore the best validation state, and only then evaluate `cifar_test_loader`.

The test result must not be used to go back and alter the search space.


In [ ]:
# TODO B28 — rebuild a fresh model from study.best_params.
# final_model = ...
# final_optimizer = ...
# final_criterion = ...

FINAL_MAX_EPOCHS = 8
FINAL_PATIENCE = 3

# TODO B29 — train while tracking the best validation state_dict.
# best_state = None
# best_val_acc = 0.0
# ...

# TODO B30 — restore best_state and evaluate test set ONCE.
# final_test_loss, final_test_acc = evaluate(...)
# print(...)


## B7 — Save and reload the final model

Save enough information to reproduce the model later:
- state dictionary;
- best hyperparameters;
- class names;
- validation/test metrics;
- optional notes about preprocessing.


In [ ]:
CHECKPOINT_PATH = "cifar10_studentcnn_optuna_best.pt"

# TODO B31 — save a checkpoint dictionary with torch.save(...).

# TODO B32 — reload the checkpoint into a newly constructed model and
# verify that evaluation gives the same result (up to numerical precision).


## B8 — Final comparison and conclusions

Create a compact comparison table using **validation metrics** for model selection:

| Approach | Model | Trainable parameters | Validation accuracy | Notes |
|---|---|---:|---:|---|
| From scratch | Your CNN | ... | ... | ... |
| Transfer learning | frozen backbone | ... | ... | ... |
| Transfer learning | partial fine-tuning | ... | ... | ... |
| Optuna | tuned StudentCNN | ... | ... | ... |

Then report the **single final test result** for the Optuna-selected model.

### Final questions
1. Which approach gave the best validation result, and at what computational cost?
2. What did transfer learning provide that your randomly initialized CNN did not?
3. Which Optuna hyperparameters were most influential in your study?
4. Which interactions between hyperparameters did you observe?
5. Why would a larger number of Optuna trials make your conclusions more reliable?
6. What would you change if accuracy were the only goal? What would you change if latency or memory were also important?
